# GT problem scan -- where vector ground truth and the rendered page disagree

Searches a list of PDFs for every problem class found while debugging `DeepVectoriser.prep_dataset` and the
raster-to-vector benchmark GT, and shows where each one occurs. The scan uses the GT builder's own parts
(`Evaluation/Labelling/visible_geometry.py`: render, `RasterText` colour split, paint rule, dense visibility
runs), so it sees what the GT sees. Logic: `rastervec/notebooks/_gt_problem_scan_helpers.py`.

Output: `outputs/gt_problem_scan/<timestamp>/` -- `summary.csv`, `findings.csv` (one row per finding, page-space
bbox in pt), `pages/<stem>_p<N>.png` overlays, `gallery/<category>.png` example crops.

### Categories

**The old paint-blind GT was wrong here** (the visible GT now handles them -- these show what changed):

| category | meaning |
|---|---|
| `paper_paint_near_ink` | path the colour split reads as paper -- white, or a light fill that may still look tinted -- within 2 px of real ink; its outline was a GT stroke before (it isn't in the training masks, so it's dropped now) |
| `light_fill` | fill-only path lighter than ink but not white -- the old DBSCAN split made it a solid "ink" layer |
| `clipped` | path cut by its clip region -- the old vote kept or dropped the whole stroke |
| `dashed` | dashed path -- the old GT drew it as one solid stroke |
| `occluded` | path cut or hidden by later paint -- the old GT kept the hidden part or dropped the visible one. At 150 dpi, tips of tiny glyphs (< 1 px) can appear with 1-4 pieces: resolution noise, ranked last; scan at 300 dpi to confirm |

**Harmless:** `paper_paint` -- paths the split reads as paper, nowhere near ink (dropped, as before).

**No vector GT can exist** (ink in the training masks with no stroke): `unbacked_text` (native font text),
`unbacked_image` (embedded images), `unbacked_other` (shadings, annotations, Type3 glyphs, anything else).

**Colour-split blind spot:** `lost_dark_ink` -- dark pixels the `RasterText` split labels paper. Its Otsu rule
can be pulled between light grey and white by a large light fill; the ink then vanishes from the masks *and*
the GT (and from inference's layers). Whether it happens depends on fill share and dpi, so scan at the dpi
you train / infer at to be sure.

For an SSIM / PSNR view of the same pages see `gt_fidelity_showcase.ipynb`.

## 0 - Configuration

In [ ]:
PDF_PATHS = [
    '../../references/test_pdfs_1.pdf',
]
PAGES = None              # list of page indices applied to every PDF, or None = every page
MAX_PAGES_PER_PDF = 3     # None = no cap
DPI = 150                 # scan resolution (the GT builder runs at 300; 150 is ~4x cheaper)
CATEGORIES = None         # e.g. ['paper_paint_near_ink', 'clipped', 'lost_dark_ink']; None = all
GALLERY_N = 12            # example crops per category
SHOW_PAGES = True         # draw each page overlay inline (always saved to disk)

## 1 - Bootstrap

In [ ]:
import os, sys

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

from IPython.display import Image, Markdown, display

from rastervec.notebooks import _gt_problem_scan_helpers as h

run_dir = h.new_run_dir()
show = [c for c in (CATEGORIES or h.CATEGORIES)]
print('output:', run_dir)

## 2 - Scan

One line per page: path count, colour layers, the split's Otsu L threshold (`none` = no lightness-based ink at
all, only chromatic -- a warning sign), then the findings per category.

In [ ]:
scans = []
for pdf in PDF_PATHS:
    for page in h.page_indices(pdf, PAGES, MAX_PAGES_PER_PDF):
        s = h.scan_page(pdf, page, dpi=DPI)
        scans.append(s)
        print(h.one_line(s))
        png = h.save_fig(h.page_overlay(s, show), run_dir / 'pages' / f'{os.path.splitext(os.path.basename(pdf))[0]}_p{page}.png')
        if SHOW_PAGES:
            display(Image(filename=str(png)))

## 3 - Summary (counts per page, totals per PDF and overall)

In [ ]:
rows = h.summary_rows(scans)
display(Markdown(h.markdown_table(rows)))
h.save_csv(rows, run_dir / 'summary.csv', h.SUMMARY_COLUMNS)
print('saved', h.save_csv(h.findings_rows(scans), run_dir / 'findings.csv', h.FINDING_COLUMNS))

## 4 - Examples per category

The highest-scoring findings of each category across every scanned page (score = pieces affected for
vector categories, ink pixels for ink categories, bbox area for paint categories), finding outlined.

In [ ]:
for cat in show:
    fig = h.gallery(scans, cat, n=GALLERY_N)
    if fig is None:
        print(f'{cat}: none found')
        continue
    display(Image(filename=str(h.save_fig(fig, run_dir / 'gallery' / f'{cat}.png'))))

## 5 - Drill down

`findings.csv` has every finding with its page-space bbox; filter it here, e.g. the clipped paths of one page.

In [ ]:
CATEGORY, PDF_NAME, PAGE = 'clipped', None, None
sel = [r for r in h.findings_rows(scans)
       if r['category'] == CATEGORY and (PDF_NAME is None or r['pdf'] == PDF_NAME) and (PAGE is None or r['page'] == PAGE)]
display(Markdown(h.markdown_table(sorted(sel, key=lambda r: -r['score'])[:50], h.FINDING_COLUMNS) if sel else f'no {CATEGORY} findings'))